In [1]:
import os
import json
import textwrap
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt

# Setup file paths
logs_dir = '../dataset/' # Adjust directory as needed
jsonl_file_path = os.path.join(logs_dir, 'experiment_logs.jsonl')

now = dt.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
pdf_output_path = os.path.join(logs_dir, f'jsonl_experiments_report_{now}.pdf')

# Read JSONL file into a list of records
records = []
if os.path.exists(jsonl_file_path):
    with open(jsonl_file_path, 'r') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
else:
    print(f"Error: Could not find '{jsonl_file_path}'")
    exit()

# Open a multi-page PDF context
with PdfPages(pdf_output_path) as pdf:
    for idx, run in enumerate(records, start=1):
        timestamp = run.get('timestamp', 'N/A')
        accuracy = run.get('accuracy', 0.0)
        hyperparams = run.get('hyperparameters', {})
        cm_dict = run.get('confusion_matrix_pct', {})
        note = textwrap.fill(str(run.get('note') or 'N/A'), width=55)

        # Extract ordered labels and rebuild 2D Confusion Matrix array
        labels = list(cm_dict.keys())
        matrix_size = len(labels)
        cm_normalized = np.zeros((matrix_size, matrix_size))

        for i, actual in enumerate(labels):
            for j, pred in enumerate(labels):
                cm_normalized[i, j] = cm_dict[actual][pred]

        # Setup Figure Layout (A4 size portrait: 8.5 x 11 inches)
        fig = plt.figure(figsize=(8.5, 11))
        
        # Title Block
        plt.suptitle(
            f"Run #{idx} | Logged: {timestamp}\n"
            f"Overall Accuracy: {accuracy * 100:.2f}% ({accuracy:.4f})",
            fontsize=12,
            fontweight='bold',
            y=0.96
        )

        # Subplot 1: Normalized Confusion Matrix Heatmap
        ax1 = fig.add_subplot(2, 1, 1)
        
        annot_labels = np.array([
            [f"{val:.1f}%" for val in row] for row in cm_normalized
        ])

        sns.heatmap(
            cm_normalized,
            annot=annot_labels,
            fmt="",
            cmap='Blues',
            xticklabels=labels,
            yticklabels=labels,
            cbar=True,
            cbar_kws={'label': 'Percentage (%)'},
            vmin=0,
            vmax=100,
            ax=ax1
        )
        
        ax1.set_title('Normalized Confusion Matrix (%)', fontsize=11, fontweight='bold', pad=10)
        ax1.set_xlabel('Predicted Activity', fontsize=9, fontweight='bold')
        ax1.set_ylabel('Actual Activity', fontsize=9, fontweight='bold')
        plt.setp(ax1.get_xticklabels(), rotation=30, ha='right', fontsize=8)
        plt.setp(ax1.get_yticklabels(), rotation=0, fontsize=8)

        # Subplot 2: Hyperparameters & Run Metadata Table
        ax2 = fig.add_subplot(2, 1, 2)
        ax2.axis('off')

        table_data = [
            ['Timestamp', str(timestamp)],
            ['Overall Accuracy', f"{accuracy:.4f} ({accuracy * 100:.2f}%)"]
        ]
        
        for param_name, param_val in hyperparams.items():
            table_data.append([f"Param: {param_name}", str(param_val)])
        table_data.append(['Note', note])
        note_row = len(table_data)

        table = ax2.table(
            cellText=table_data,
            colLabels=['Parameter / Metric', 'Value'],
            loc='center',
            cellLoc='left'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.0, 1.4)

        # Give the wrapped note enough height to remain readable.
        note_height = table[(note_row, 1)].get_height() * len(note.splitlines())
        for col_idx in range(2):
            table[(note_row, col_idx)].set_height(note_height)

        # Highlight header row
        for (row_idx, col_idx), cell in table.get_celld().items():
            if row_idx == 0:
                cell.set_facecolor('#d3d3d3')
                cell.get_text().set_weight('bold')

        ax2.set_title('Model Hyperparameters & Metadata', fontsize=11, fontweight='bold', pad=10)

        plt.tight_layout(rect=[0, 0.03, 1, 0.93])
        
        # Append page to PDF
        pdf.savefig(fig)
        plt.close(fig)

print(f"Successfully generated multi-page evaluation PDF at: {pdf_output_path}")

Successfully generated multi-page evaluation PDF at: ../dataset/jsonl_experiments_report_2026-10-02_11-29-37.pdf
